<a href="https://colab.research.google.com/github/abdulwahab-git/week-01-Assignment/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwahab-git/week-01-Assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one (client, content item, day)" confirm the HAVING query returns 0 rows. State the actual earliest/latest date and day count once you run it.*

In [ ]:
con.sql("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────┬───────┐
│     client_hash_id      │     content_hash_id      │ report_date │   n   │
│         varchar         │         varchar          │    date     │ int64 │
├─────────────────────────┼──────────────────────────┼─────────────┼───────┤
│ client_1a730cb2640a1abf │ content_6604767cde89152e │ 2026-06-13  │     2 │
│ client_1a730cb2640a1abf │ content_b5aec9a8a2ee7fb0 │ 2026-06-13  │     2 │
│ client_b77d0d5f08f05e64 │ content_1b18ea04806f1278 │ 2026-06-13  │     2 │
│ client_8ddc46da5414ffd8 │ content_a9789f58505f4f9b │ 2026-06-13  │     2 │
│ client_8ddc46da5414ffd8 │ content_d07f96c572dc6f2a │ 2026-06-13  │     2 │
└─────────────────────────┴──────────────────────────┴─────────────┴───────┘

In [ ]:
con.sql("""
    SELECT MIN(report_date) AS earliest, MAX(report_date) AS latest, COUNT(DISTINCT report_date) AS n_days
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────┐
│  earliest  │   latest   │ n_days │
│    date    │    date    │ int64  │
├────────────┼────────────┼────────┤
│ 2025-01-27 │ 2026-06-30 │    520 │
└────────────┴────────────┴────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
    DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    LIMIT 0
""")
con.sql("""
    DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
    LIMIT 0
""")

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
con.sql("""
    SELECT
        SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS null_clicks,
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_impr,
        SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS null_ga4_sessions,
        COUNT(*) AS total
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬───────────┬───────────────────┬──────────┐
│ null_clicks │ null_impr │ null_ga4_sessions │  total   │
│   int128    │  int128   │      int128       │  int64   │
├─────────────┼───────────┼───────────────────┼──────────┤
│       98006 │     98006 │          29635327 │ 78835655 │
└─────────────┴───────────┴───────────────────┴──────────┘

In [ ]:
con.sql("""
    SELECT
        SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS null_clicks,
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_impr,
        SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS null_ga4_sessions,
        COUNT(*) AS total
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
""")

con.sql("""
    SELECT client_hash_id, COUNT(*) AS rows, MIN(report_date), MAX(report_date)
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    GROUP BY 1 ORDER BY 2 DESC LIMIT 10
""")

# how sparse is AI-referral tracking?
con.sql("""
    SELECT
        SUM(CASE WHEN sessions_ai > 0 THEN 1 ELSE 0 END) AS rows_with_ai_traffic,
        COUNT(*) AS total
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────┬──────────┐
│ rows_with_ai_traffic │  total   │
│        int128        │  int64   │
├──────────────────────┼──────────┤
│                30177 │ 78835655 │
└──────────────────────┴──────────┘

actual null percentages for gsc_clicks/gsc_impressions (expect near-zero if client_has_gsc), much higher null rate for ga4_sessions (only ~51 of 104 clients have GA4 per section 4), and the actual % of rows with nonzero AI-referral traffic — likely very small since this is a new tracking category.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df = con.sql("""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
""").df()

import pandas as pd
df['gsc_data_start'] = pd.to_datetime(df['gsc_data_start'])
df['ga4_data_start'] = pd.to_datetime(df['ga4_data_start'])

# get the latest date actually present in the daily fact table
max_date = con.sql("""
    SELECT MAX(month) FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
""").fetchone()[0]

df['gsc_months'] = (pd.Timestamp(max_date) - df['gsc_data_start']).dt.days / 30
df['gap_days'] = (df['ga4_data_start'] - df['gsc_data_start']).dt.days

print("--- access_profile counts ---")
print(df['access_profile'].value_counts())
print("\n--- gsc_months (history depth) ---")
print(df['gsc_months'].describe())
print("\n--- gap_days (GSC start to GA4 start) ---")
print(df['gap_days'].describe())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- access_profile counts ---
access_profile
gsc_and_ga4                             53
no_search_or_analytics_access           26
gsc_only                                14
source_only_missing_client_dimension    10
ga4_only                                 1
Name: count, dtype: int64

--- gsc_months (history depth) ---
count    67.000000
mean      6.532338
std       4.157444
min      -0.033333
25%       3.400000
50%       6.933333
75%       8.333333
max      16.333333
Name: gsc_months, dtype: float64

--- gap_days (GSC start to GA4 start) ---
count     49.000000
mean      92.142857
std      111.310751
min      -23.000000
25%        3.000000
50%       27.000000
75%      150.000000
max      406.000000
Name: gap_days, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.